This is the first part of a talk for GOSIM 2026, given together with Olivier Grisel (who
does the second half of this talk).

## The Adoption of Python Array API in scikit-learn

### What is the goal of introducing Array API in scikit-learn
- scikit-learn is seasoned a machine learning library that was build with running on
  numpy arrays on CPUs in mind
- with more people using GPUs for ML, more libraries come into play
- since version 1.3 (release in June 2023) we implement support in a growing number or
  functions and estimators in experimental mode
    - see [Array API support
      (experimental)](https://scikit-learn.org/stable/modules/array_api.html) for the
      docs on supported elements

- in the Python ML ecosystem, scikit-learn is an array consuming library
    - a library that gets arrays as inputs and outputs arrays
    - the goal is that during the operations within scikit-learn, the arrays stay on the
      namespace and on the device (specifically GPU devices) they came from for all of
      the computationally intense tasks and for most of the tasks in general
    - as a consequence, users in addition to numpy array can pass CuPy arrays and
      PyTorch tensors 


### Implementation
- `array_api_compat` bridges gaps for parts of numpy, torch and cupy, that don't yet
  fully support the array API --> scikit-learn depended on it in versions 1.3-???, now a
  fixed version of `array_api_compat` is vendored inside scikit-learn

- the Array API standard, developed by the Consortium for Python Data API Standards
  (consisting of maintainers of array heavy Python libraries), defines the array
  API as a limited subset of numpy functionality
- the Array API standard defines data types, operations and behaviours
  - limited subset of types and operations; for instance no support for sparse arrays
    or matrixes
  - not the whole width of functionality; for instance no u-funcs, sometimes need to
    re-write computational logic for arrays other than numpy

  - some higher order functions are implemented in scikit-learn, mostly by subcasing the
    behavior depending on the different backends and where the function is build from
    lower order function within the array API spec; for instance `_nanmin()`

  - for remaining functions, `array-api-extra` provides array functions not included in
    the standard but deemed useful for array-consuming libraries; it recently got also
    vendored within scikit-learn

- there is the testing library `array-api-strict` that mimics a library that solely
 emulates the array API standard that we use to check if we support the min standard
  - even though torch and CuPy mostly support more functionality - that makes it
      possible for further array libraries to implement minimal support 
  - `array-api-strict` still changes (slightly) and is open for future enhancement
        when support for more functions may be targeted

- 

### Usage
- since scipy has array API support in experimental mode as well, users need to set `export SCIPY_ARRAY_API=1`
- users need to enable array API support by setting `array_api_dispatch=True``from sklearn import set_config
set_config(`array_api_dispatch=True`)`
or by using our config context manager:

In [ ]:
# export SCIPY_ARRAY_API=1 # <-- set environment variable

from sklearn.datasets import make_classification
from sklearn import config_context
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
import cupy

X_np, y_np = make_classification(random_state=0)
X_cu = cupy.asarray(X_np)
y_cu = cupy.asarray(y_np)
X_cu.device

with config_context(array_api_dispatch=True):
    lda = LinearDiscriminantAnalysis()
    X_trans = lda.fit_transform(X_cu, y_cu)
X_trans.device

### Challenges
- as a consuming library, scikit-learn depends on upstream implementations, from numpy
  and scipy; especially scipy is blocking us sometimes
- plans of moving array API support in scikit-learn out of experimental mode are in the
  making
  - specifically we need to wait for scipy to move out of experimental, since sklearn
    should not turn on SciPy's experimental, process-wide mode on the user's behalf and
    a non-experimental mode in scikit-learn needs a stable non-surprising usage of
    different array types in scipy
- no plans so far to add support for scikit-learn's compiled code written in Cython, C
  or C++ (only runs on CPUs), since the Python Array API is a python project
    - the challenge on how to deal with estimators that do the heavy part of the
      computation on compiled code is not fully discovered and not solved
    - for now, unsupported parts of the code are dealt with by converting (copying) the
      relevant arrays to numpy and transferring them to CPU, doing the compiled
      computation and then converting them back to the input namespace and device
    - discovering if plugins can be used as alternatives to the CPU optimised current
      Cython code


### References
- Lucy Liu: [Update on array API adoption in scikit-learn](https://labs.quansight.org/blog/array-api-scikit-learn-2026), Quansight Labs Blog, March 6 2026.
- Thomas J. Fan: [Array API Support in scikit-learn](https://labs.quansight.org/blog/array-api-support-scikit-learn), Quansight Labs Blog, September 19 2023.